# Python Fundamentals Summary — TSM Project (Stage 03)

This notebook demonstrates the core Python, NumPy, and pandas skills the project
will build on, using **dummy data** (no real market data yet). It also imports and
exercises the reusable functions in `src/utils.py`, which later stages will reuse.


In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                    # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))     # so `from src....` imports work
print('working from:', ROOT.name)

working from: project


## 1) Python fundamentals

Plain-Python building blocks: data structures and a small reusable function.

In [2]:
# Lists, dicts, and a small reusable function (pure Python)
prices = [150.0, 151.2, 149.8, 152.4, 153.0]
ticker_info = {'ticker': 'TSM', 'exchange': 'NYSE', 'currency': 'USD'}

def daily_return(prev, curr):
    return (curr - prev) / prev

# Compute pairwise daily returns with a list comprehension
rets = [daily_return(prices[i - 1], prices[i]) for i in range(1, len(prices))]
print('ticker:', ticker_info['ticker'])
print('daily returns:', [round(r, 4) for r in rets])

ticker: TSM
daily returns: [0.008, -0.0093, 0.0174, 0.0039]


## 2) NumPy fundamentals

Vectorised math on arrays — the basis for the feature engineering later.

In [3]:
import numpy as np

arr = np.array(prices)
log_returns = np.diff(np.log(arr))            # vectorised, no Python loop
print('log returns:', np.round(log_returns, 4))
print('mean:', round(float(np.mean(log_returns)), 4))
print('std :', round(float(np.std(log_returns, ddof=1)), 4))   # sample std

log returns: [ 0.008  -0.0093  0.0172  0.0039]
mean: 0.005
std : 0.011


## 3) pandas fundamentals

Build a small DataFrame and run basic operations on it.

In [4]:
import pandas as pd

# Dummy daily OHLCV for a made-up ticker
dates = pd.date_range('2024-01-01', periods=5, freq='D')
df = pd.DataFrame({
    'Date': dates,
    'Close': [100.0, 101.5, 101.0, 102.0, 103.5],
    'Volume': [1_000_000, 1_200_000, 950_000, 1_100_000, 1_300_000],
})
print(df)
print(df[['Close', 'Volume']].describe().round(2))

        Date  Close   Volume
0 2024-01-01  100.0  1000000
1 2024-01-02  101.5  1200000
2 2024-01-03  101.0   950000
3 2024-01-04  102.0  1100000
4 2024-01-05  103.5  1300000
        Close      Volume
count    5.00        5.00
mean   101.60  1110000.00
std      1.29   143178.21
min    100.00   950000.00
25%    101.00  1000000.00
50%    101.50  1100000.00
75%    102.00  1200000.00
max    103.50  1300000.00


## 4) Reusable utilities from `src/utils.py`

Apply the project's own helpers to a (cleaned) version of the dummy frame, to
confirm they work and to preview the shape of the real pipeline.

In [5]:
from src.utils import clean_column_names, ensure_datetime_index, add_returns

# Simulate raw yfinance-style column names
raw = pd.DataFrame({
    'Date': dates,
    'Adj Close': [100.0, 101.5, 101.0, 102.0, 103.5],
    'Volume': [1_000_000, 1_200_000, 950_000, 1_100_000, 1_300_000],
})

clean = clean_column_names(raw)          # 'Adj Close' -> 'adj_close'
clean = ensure_datetime_index(clean, col='date')
clean = add_returns(clean, close_col='adj_close')

print(clean[['adj_close', 'volume', 'return']].round(4))
print('columns:', list(clean.columns))
print('index is datetime:', isinstance(clean.index, pd.DatetimeIndex))

            adj_close   volume  return
date                                  
2024-01-01      100.0  1000000     NaN
2024-01-02      101.5  1200000  0.0150
2024-01-03      101.0   950000 -0.0049
2024-01-04      102.0  1100000  0.0099
2024-01-05      103.5  1300000  0.0147
columns: ['adj_close', 'volume', 'return']
index is datetime: True
